In [ ]:
!pip install -q timm pytorch-metric-learning

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.8/127.8 kB 4.4 MB/s eta 0:00:00


In [ ]:
# ============================================================
# STRONG METRIC LEARNING TRAINING PIPELINE
# ArcFace + MultiSimilarity (stable configuration)
# Aspect-ratio preserving pipeline
# RP2K pretrained EfficientNetV2-S
# Optimized for low-FMR verification
# ============================================================

import os
import gc
import cv2
import math
import random
import warnings
from pathlib import Path

import albumentations as A
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import timm

from albumentations.pytorch import ToTensorV2
from PIL import Image
from pytorch_metric_learning import losses, samplers
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from torch.utils.data import Dataset, DataLoader
from tqdm.auto import tqdm

warnings.filterwarnings("ignore")

# ============================================================
# CONFIG
# ============================================================

DATASET_PATH = Path(
    "/kaggle/input/competitions/dl-lab-5-metric-learning/train/train"
)

RP2K_PATH = ("/kaggle/input/models/meowmeowmeeowww/enet-rp2k/transformers/default/1/efficientnetv2s_rp2k.pth")

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

SEED = 42

TARGET_SIZE = 256
EMBEDDING_SIZE = 512

BATCH_SIZE = 64
ACCUM_STEPS = 2
EFFECTIVE_BATCH = BATCH_SIZE * ACCUM_STEPS

NUM_WORKERS = 4

# ---------------------------
# TRAINING PLAN
# ---------------------------

NUM_EPOCHS = 18

BACKBONE_LR = 2e-4
HEAD_LR = 5e-4

WEIGHT_DECAY = 1e-4

WARMUP_EPOCHS = 2
MIN_LR = 1e-6

# ---------------------------
# ArcFace
# ---------------------------

MARGIN = 0.5
SCALE = 64

# ---------------------------
# MultiSimilarity
# ---------------------------

MS_ALPHA = 2.0
MS_BETA = 50.0
MS_BASE = 0.55

PAIR_WEIGHT = 0.15

# ---------------------------
# Validation
# ---------------------------

FMR_LEVELS = [1e-3, 1e-4]

# ============================================================
# REPRODUCIBILITY
# ============================================================

torch.manual_seed(SEED)
np.random.seed(SEED)
random.seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.benchmark = True
    torch.backends.cudnn.deterministic = False

# ============================================================
# NORMALIZATION
# ============================================================

MEAN = [0.485, 0.456, 0.406]
STD = [0.229, 0.224, 0.225]

# ============================================================
# TRANSFORMS
# ============================================================

def make_train_transform(target_size=256):

    return A.Compose([

        # Preserve aspect ratio
        A.LongestMaxSize(
            max_size=target_size,
            interpolation=cv2.INTER_CUBIC
        ),

        # Pad to square
        A.PadIfNeeded(
            min_height=target_size,
            min_width=target_size,
            border_mode=cv2.BORDER_CONSTANT,
            fill=(0, 0, 0),
            position="center"
        ),

        # Mild geometry only
        A.HorizontalFlip(p=0.5),

        # Compression artifacts
        A.OneOf([
            A.ImageCompression(
                quality_range=(40, 100),
                compression_type="jpeg",
                p=1.0
            ),
            A.ImageCompression(
                quality_range=(40, 100),
                compression_type="webp",
                p=1.0
            )
        ], p=0.3),

        # Mild photometric aug
        A.ColorJitter(
            brightness=0.2,
            contrast=0.2,
            saturation=0.2,
            hue=0.05,
            p=0.5
        ),

        # Small blur occasionally
        A.GaussianBlur(
            blur_limit=(3, 5),
            p=0.10
        ),

        # Very mild erasing
        A.CoarseDropout(
            num_holes_range=(1, 2),
            hole_height_range=(0.03, 0.10),
            hole_width_range=(0.03, 0.10),
            fill=0,
            p=0.15
        ),

        A.Normalize(mean=MEAN, std=STD),

        ToTensorV2()
    ])


def make_val_transform(target_size=256):

    return A.Compose([

        A.LongestMaxSize(
            max_size=target_size,
            interpolation=cv2.INTER_CUBIC
        ),

        A.PadIfNeeded(
            min_height=target_size,
            min_width=target_size,
            border_mode=cv2.BORDER_CONSTANT,
            fill=(0, 0, 0),
            position="center"
        ),

        A.Normalize(mean=MEAN, std=STD),

        ToTensorV2()
    ])


train_transform = make_train_transform(TARGET_SIZE)
val_transform = make_val_transform(TARGET_SIZE)

# ============================================================
# DATA PREPARATION
# ============================================================

print("Loading data...")

all_images = []
all_labels = []

plu_dirs = [p for p in DATASET_PATH.iterdir() if p.is_dir()]

for plu_dir in plu_dirs:

    label = plu_dir.name

    for img_path in plu_dir.iterdir():

        if img_path.suffix.lower() in [".jpg", ".jpeg", ".png"]:

            all_images.append(str(img_path))
            all_labels.append(label)

print(f"Images: {len(all_images)}")
print(f"Classes: {len(plu_dirs)}")

label_encoder = LabelEncoder()
all_labels_enc = label_encoder.fit_transform(all_labels)

num_classes = len(label_encoder.classes_)

unique_classes = np.unique(all_labels_enc)

train_classes, val_classes = train_test_split(
    unique_classes,
    test_size=0.20,
    random_state=SEED
)

train_mask = np.isin(all_labels_enc, train_classes)
val_mask = np.isin(all_labels_enc, val_classes)

train_imgs = np.array(all_images)[train_mask].tolist()
train_labels = all_labels_enc[train_mask]

val_imgs = np.array(all_images)[val_mask].tolist()
val_labels = all_labels_enc[val_mask]

print(f"Train classes: {len(train_classes)}")
print(f"Val classes: {len(val_classes)}")

print(f"Train images: {len(train_imgs)}")
print(f"Val images: {len(val_imgs)}")

# ============================================================
# DATASET
# ============================================================

class PLUDataset(Dataset):

    def __init__(self, image_paths, labels, transform=None):

        self.image_paths = image_paths
        self.labels = labels
        self.transform = transform

    def __len__(self):

        return len(self.image_paths)

    def __getitem__(self, idx):

        img = Image.open(self.image_paths[idx]).convert("RGB")

        img = np.array(img)

        label = self.labels[idx]

        if self.transform is not None:

            img = self.transform(image=img)["image"]

        return img, label


train_dataset = PLUDataset(
    train_imgs,
    train_labels,
    transform=train_transform
)

val_dataset = PLUDataset(
    val_imgs,
    val_labels,
    transform=val_transform
)

# ============================================================
# SAMPLER / DATALOADER
# ============================================================

sampler = samplers.MPerClassSampler(
    labels=train_labels,
    m=4,
    length_before_new_iter=len(train_dataset)
)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    sampler=sampler,
    num_workers=NUM_WORKERS,
    pin_memory=True,
    persistent_workers=True,
    prefetch_factor=2,
    drop_last=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE * 2,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True,
    persistent_workers=True,
    prefetch_factor=2
)

# ============================================================
# MODEL
# ============================================================

class MetricModel(nn.Module):

    def __init__(
        self,
        backbone_name,
        embedding_size,
        pretrained_path=None
    ):

        super().__init__()

        self.backbone = timm.create_model(
            backbone_name,
            pretrained=False,
            num_classes=0
        )

        if pretrained_path is not None and os.path.exists(pretrained_path):

            print(f"\nLoading RP2K weights:\n{pretrained_path}")

            ckpt = torch.load(
                pretrained_path,
                map_location="cpu"
            )

            if "backbone_state_dict" in ckpt:

                sd = ckpt["backbone_state_dict"]

            else:

                sd = {
                    k: v for k, v in ckpt.items()
                    if not k.startswith("classifier")
                }

            sd = {
                k.replace("_orig_mod.", "").replace("classifier.", ""): v
                for k, v in sd.items()
            }

            missing, unexpected = self.backbone.load_state_dict(
                sd,
                strict=False
            )

            print("Missing keys:", missing)
            print("Unexpected keys:", unexpected)

        else:

            print("Using ImageNet pretrained")

            self.backbone = timm.create_model(
                backbone_name,
                pretrained=True,
                num_classes=0
            )

        in_features = self.backbone.num_features

        self.embedding = nn.Sequential(

            nn.Linear(
                in_features,
                embedding_size,
                bias=False
            ),

            nn.BatchNorm1d(embedding_size)

        )

    def forward(self, x):

        feats = self.backbone(x)

        embs = self.embedding(feats)

        embs = F.normalize(embs, p=2, dim=1)

        return embs


model = MetricModel(
    "tf_efficientnetv2_s_in21ft1k",
    EMBEDDING_SIZE,
    pretrained_path=RP2K_PATH
).to(DEVICE)

model = model.to(memory_format=torch.channels_last)

# ============================================================
# LOSSES
# ============================================================

arcface_loss = losses.ArcFaceLoss(
    num_classes=num_classes,
    embedding_size=EMBEDDING_SIZE,
    margin=MARGIN,
    scale=SCALE
).to(DEVICE)

ms_loss = losses.MultiSimilarityLoss(
    alpha=MS_ALPHA,
    beta=MS_BETA,
    base=MS_BASE
)

# ============================================================
# OPTIMIZER
# ============================================================

optimizer = torch.optim.AdamW(

    [
        {
            "params": model.parameters(),
            "lr": BACKBONE_LR
        },

        {
            "params": arcface_loss.parameters(),
            "lr": HEAD_LR
        }
    ],

    weight_decay=WEIGHT_DECAY

)

# ============================================================
# SCHEDULER
# ============================================================

def cosine_warmup_scheduler(
    optimizer,
    warmup_epochs,
    total_epochs,
    min_lr=1e-6
):

    def lr_lambda(epoch):

        if epoch < warmup_epochs:

            return float(epoch + 1) / float(max(1, warmup_epochs))

        progress = (
            (epoch - warmup_epochs)
            / float(max(1, total_epochs - warmup_epochs))
        )

        cosine = 0.5 * (1.0 + math.cos(math.pi * progress))

        return max(min_lr / BACKBONE_LR, cosine)

    return torch.optim.lr_scheduler.LambdaLR(
        optimizer,
        lr_lambda
    )


scheduler = cosine_warmup_scheduler(
    optimizer,
    warmup_epochs=WARMUP_EPOCHS,
    total_epochs=NUM_EPOCHS,
    min_lr=MIN_LR
)

# ============================================================
# AMP
# ============================================================

scaler = torch.amp.GradScaler(DEVICE.type)

# ============================================================
# VALIDATION
# ============================================================

@torch.no_grad()
def validate_fnmr_robust(
    model,
    loader,
    device,
    fmr_targets=[1e-3, 1e-4]
):

    model.eval()

    all_embs = []
    all_lbls = []

    for images, labels in tqdm(loader, desc="Val"):

        images = images.to(
            device,
            memory_format=torch.channels_last,
            non_blocking=True
        )

        embs = model(images)

        all_embs.append(embs.cpu())
        all_lbls.append(labels)

    all_embs = torch.cat(all_embs).numpy()
    all_lbls = torch.cat(all_lbls).numpy()

    sim = all_embs @ all_embs.T

    pos_mask = np.equal.outer(all_lbls, all_lbls)
    np.fill_diagonal(pos_mask, False)

    neg_mask = ~pos_mask
    np.fill_diagonal(neg_mask, False)

    pos_scores = sim[pos_mask]
    neg_scores = sim[neg_mask]

    results = {}

    n_neg = len(neg_scores)

    sorted_neg = np.sort(neg_scores)[::-1]

    for fmr in fmr_targets:

        num_allow = max(1, int(fmr * n_neg))

        threshold = sorted_neg[num_allow - 1]

        fnmr = np.mean(pos_scores < threshold)

        results[fmr] = {
            "fnmr": fnmr,
            "threshold": threshold
        }

    return results

# ============================================================
# TRAINING LOOP
# ============================================================

best_score = 999.0
best_epoch = 0

for epoch in range(NUM_EPOCHS):

    print("\n" + "=" * 70)
    print(f"Epoch {epoch+1}/{NUM_EPOCHS}")

    gc.collect()
    torch.cuda.empty_cache()

    model.train()
    arcface_loss.train()

    optimizer.zero_grad()

    running_loss = 0.0

    pbar = tqdm(train_loader, desc="Training")

    for step, (images, labels) in enumerate(pbar):

        images = images.to(
            DEVICE,
            memory_format=torch.channels_last,
            non_blocking=True
        )

        labels = labels.to(
            DEVICE,
            non_blocking=True
        )

        with torch.amp.autocast(DEVICE.type):

            embs = model(images)

            loss_arc = arcface_loss(
                embs,
                labels
            )

            loss_ms = ms_loss(
                embs,
                labels
            )

            loss = (
                loss_arc
                + PAIR_WEIGHT * loss_ms
            )

            loss = loss / ACCUM_STEPS

        scaler.scale(loss).backward()

        if (step + 1) % ACCUM_STEPS == 0:

            scaler.unscale_(optimizer)

            torch.nn.utils.clip_grad_norm_(
                model.parameters(),
                max_norm=5.0
            )

            scaler.step(optimizer)
            scaler.update()

            optimizer.zero_grad()

        running_loss += loss.item() * ACCUM_STEPS

        pbar.set_postfix(
            loss=f"{loss.item() * ACCUM_STEPS:.4f}",
            arc=f"{loss_arc.item():.4f}",
            ms=f"{loss_ms.item():.4f}"
        )

    avg_loss = running_loss / len(train_loader)

    # ========================================================
    # VALIDATION
    # ========================================================

    fnmr_res = validate_fnmr_robust(
        model,
        val_loader,
        DEVICE,
        FMR_LEVELS
    )

    print(f"\nLoss: {avg_loss:.4f}")

    for fmr, res in fnmr_res.items():

        print(
            f"FMR={fmr:.0e} | "
            f"FNMR={res['fnmr']:.4f} | "
            f"thr={res['threshold']:.4f}"
        )

    val_score = (
        0.7 * fnmr_res[1e-3]["fnmr"]
        + 0.3 * fnmr_res[1e-4]["fnmr"]
    )

    if val_score < best_score:

        best_score = val_score
        best_epoch = epoch + 1

        torch.save({

            "model_state_dict": model.state_dict(),
            "epoch": epoch,
            "score": val_score

        }, "/kaggle/working/best_metric_model_512.pth")

        print("✅ Saved best model")

    scheduler.step()

    print(
        f"LR: {optimizer.param_groups[0]['lr']:.2e}"
    )

print("\nTraining finished")
print(f"Best epoch: {best_epoch}")
print(f"Best score: {best_score:.5f}")

Loading data...
Images: 13374
Classes: 1000
Train classes: 800
Val classes: 200
Train images: 10668
Val images: 2706

Loading RP2K weights:
/kaggle/input/models/meowmeowmeeowww/enet-rp2k/transformers/default/1/efficientnetv2s_rp2k.pth
Missing keys: []
Unexpected keys: ['weight', 'bias']

Epoch 1/18


Training:   0%|          | 0/150 [00:00<?, ?it/s]

Val:   0%|          | 0/22 [00:00<?, ?it/s]


Loss: 8.9413
FMR=1e-03 | FNMR=0.1089 | thr=0.4298
FMR=1e-04 | FNMR=0.2127 | thr=0.5265
✅ Saved best model
LR: 2.00e-04

Epoch 2/18


Training:   0%|          | 0/150 [00:00<?, ?it/s]

...

Val:   0%|          | 0/22 [00:00<?, ?it/s]


Loss: 0.0901
FMR=1e-03 | FNMR=0.0511 | thr=0.4657
FMR=1e-04 | FNMR=0.1296 | thr=0.5779
LR: 1.92e-06

Epoch 18/18


Training:   0%|          | 0/150 [00:00<?, ?it/s]

Val:   0%|          | 0/22 [00:00<?, ?it/s]


Loss: 0.0894
FMR=1e-03 | FNMR=0.0528 | thr=0.4653
FMR=1e-04 | FNMR=0.1334 | thr=0.5781
LR: 1.00e-06

Training finished
Best epoch: 15
Best score: 0.07407


In [ ]:
print("\nLoading best Stage 1 model for final submission...")
ckpt = torch.load('/kaggle/working/best_metric_model_512.pth', map_location='cpu', weights_only=False)
model.load_state_dict(ckpt['model_state_dict'])
model.eval()

# ---------- STEP 1: Create train loader for PCA fitting ----------
# Recreate train dataset with val_transform (no augmentations)
train_dataset_for_pca = PLUDataset(train_imgs, train_labels, transform=val_transform)
train_loader_for_pca = DataLoader(
    train_dataset_for_pca,
    batch_size=BATCH_SIZE * 2,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True
)

print("Extracting train embeddings for PCA...")
train_embs_for_pca = []
with torch.no_grad():
    for images, labels in tqdm(train_loader_for_pca, desc="Train embeddings"):
        images = images.to(DEVICE, memory_format=torch.channels_last, non_blocking=True)
        feats = model(images).cpu().numpy()
        train_embs_for_pca.append(feats)
train_embs_for_pca = np.concatenate(train_embs_for_pca)

# ---------- STEP 2: Fit PCA on TRAINING data ----------
from sklearn.decomposition import PCA

mean = train_embs_for_pca.mean(axis=0)
train_embs_centered = train_embs_for_pca - mean

pca = PCA(n_components=96, whiten=True)
pca.fit(train_embs_centered)

print(f"PCA explained variance: {pca.explained_variance_ratio_.sum():.3f}")

# Free memory
del train_embs_for_pca, train_embs_centered
gc.collect()
torch.cuda.empty_cache()

# ---------- STEP 3: Extract TEST embeddings ----------
TEST_ROOT = Path("/kaggle/input/competitions/dl-lab-5-metric-learning/test_kaggle/test_kaggle")
SUBMISSION_PATH = Path("/kaggle/input/competitions/dl-lab-5-metric-learning/submission.csv")
OUTPUT_PATH = "/kaggle/working/submission_arcface.csv"

class TestDataset(Dataset):
    def __init__(self, root, transform):
        self.files = sorted([f for f in root.rglob('*') if f.suffix.lower() in ['.jpg','.jpeg','.png']])
        self.transform = transform

    def __len__(self):
        return len(self.files)

    def __getitem__(self, idx):
        path = self.files[idx]
        img = np.array(Image.open(path).convert('RGB'))
        if self.transform:
            img = self.transform(image=img)["image"]
        return path.name, img

test_ds = TestDataset(TEST_ROOT, val_transform)
test_ldr = DataLoader(test_ds, batch_size=BATCH_SIZE * 2, shuffle=False,
                      num_workers=NUM_WORKERS, pin_memory=True)

test_embs, test_names = [], []
with torch.no_grad():
    for names, images in tqdm(test_ldr, desc="Test inference"):
        images = images.to(DEVICE, memory_format=torch.channels_last, non_blocking=True)
        feats = model(images).cpu().numpy()
        test_embs.append(feats)
        test_names.extend(names)

test_embs = np.concatenate(test_embs)

# ---------- STEP 4: Apply fitted PCA to TEST embeddings ----------
test_embs_centered = test_embs - mean  # Use TRAIN mean
test_embs_pca = pca.transform(test_embs_centered)  # Use fitted PCA
test_embs_pca = test_embs_pca / (np.linalg.norm(test_embs_pca, axis=1, keepdims=True) + 1e-8)
test_embs = test_embs_pca

# ---------- STEP 5: Compute similarities ----------
name2emb = {n: e for n, e in zip(test_names, test_embs)}

import pandas as pd
sub = pd.read_csv(SUBMISSION_PATH)
if 'similarity' in sub.columns:
    sub = sub.drop(columns=['similarity'])

sims = []
missing_files = []

for _, row in tqdm(sub.iterrows(), total=len(sub), desc="Similarities"):
    f1, f2 = row['file_1'], row['file_2']

    if f1 not in name2emb or f2 not in name2emb:
        missing_files.append((f1, f2))
        sims.append(0.0)
        continue

    sims.append(float(np.dot(name2emb[f1], name2emb[f2])))

if missing_files:
    print(f"⚠️ {len(missing_files)} pairs had missing files, using similarity=0.0")

sub['similarity'] = sims
sub.to_csv(OUTPUT_PATH, index=False)

print(f"✅ Saved to {OUTPUT_PATH}")
print(f"Similarity range: [{min(sims):.4f}, {max(sims):.4f}], mean={np.mean(sims):.4f}")